# Using an LLM as an Agent

This notebook shows the core pattern: pass **evidence** (real data) into a prompt, and let the LLM reason about it.

No tool calls, no frameworks — just a system prompt, a user prompt with data, and a response.

**Key concepts:**
- **System prompt** — tells the LLM its role and how to behave
- **User prompt** — contains the actual data (evidence-based prompting)
- **Few-shot examples** — teach the model the exact output format you expect
- **Token usage** — every call costs tokens; understanding limits matters
- **Parallel calls** — run multiple agents concurrently with `asyncio`

**Requirements:** `pip install openai python-dotenv` and add your `OPENAI_API_KEY` to the `.env` file in the project root.

In [ ]:
import os
import json
import asyncio
import time
from dotenv import load_dotenv
from openai import OpenAI, AsyncOpenAI

# Load API keys from .env file in the project root
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), ".env"))

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
async_client = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

MIN_COMPLETION_TOKENS = 800

# ---------------------------------------------------------------------------
# SYSTEM PROMPT — the agent's role, instructions, and few-shot examples
#
# Few-shot examples teach the model the EXACT output format and reasoning
# depth you expect. Without them the model guesses at structure; with them
# it mirrors the pattern consistently.
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = """\
You are a promotional strategy analyst for a grocery retailer.
Given item-level promotion data, respond with a JSON object containing:

  sku                - the SKU analyzed
  profitable         - boolean, was this promotion profitable?
  incremental_profit - dollar amount of incremental profit (or loss)
  recommendation     - one of: "repeat", "modify", "discontinue"
  next_steps         - list of specific actions to take
  future_changes     - list of ways to apply this promotion differently
  reasoning          - brief explanation supporting your recommendation

Be specific. Use the numbers provided. Do not fabricate data.

## FEW-SHOT EXAMPLES

### Example 1 — Profitable promotion → repeat with tweaks

INPUT:
SKU: DAI-10432 | Product: Greek Yogurt 32oz | Category: Dairy
Baseline: 210 units/wk, 32% margin
Promo: 20% Off, 2 weeks, +45% lift (305 units/wk), 24% margin, $600 spend
Cannibalization: 5% | Post-promo: 195 units (near baseline)

OUTPUT:
{
  "sku": "DAI-10432",
  "profitable": true,
  "incremental_profit": 312.40,
  "recommendation": "repeat",
  "next_steps": [
    "Schedule repeat for Q3 back-to-school period when yogurt demand peaks",
    "Negotiate co-op funding with supplier to offset $600 ad spend",
    "Track 4-week post-promo demand to confirm no lasting forward-buy effect"
  ],
  "future_changes": [
    "Test a 15% discount instead of 20% to preserve more margin",
    "Extend to 3 weeks to capture more incremental volume",
    "Cross-merchandise with granola to reduce cannibalization within dairy"
  ],
  "reasoning": "Incremental volume of 190 units over 2 weeks at 24% margin generated $364 gross profit. After subtracting $600 promo spend and adjusting for 5% cannibalization, net incremental profit is $312.40. Post-promo demand returned near baseline with minimal forward-buying, confirming real demand lift."
}

### Example 2 — Unprofitable promotion → discontinue

INPUT:
SKU: BEV-77210 | Product: Sparkling Water 12-Pack | Category: Beverages
Baseline: 88 units/wk, 28% margin
Promo: Buy 2 Get 1 Free, 4 weeks, +120% lift (194 units/wk), 9% margin, $2,400 spend
Cannibalization: 22% | Post-promo: 51 units (well below baseline for 3 weeks)

OUTPUT:
{
  "sku": "BEV-77210",
  "profitable": false,
  "incremental_profit": -1874.50,
  "recommendation": "discontinue",
  "next_steps": [
    "Remove Buy 2 Get 1 Free from future promotional calendar for this SKU",
    "Monitor next 4 weeks to see when demand recovers to baseline",
    "Assess shelf-stable inventory levels — customers likely stockpiled"
  ],
  "future_changes": [
    "If revisiting, test a smaller discount (e.g. 10% off single unit) to avoid pantry-loading",
    "Limit to 1-week duration to reduce forward-buy window",
    "Restrict to loyalty card holders only to prevent cherry-picking"
  ],
  "reasoning": "Despite +120% lift, margin collapsed from 28% to 9%. Incremental gross profit from 424 extra units over 4 weeks was only $525.50. After $2,400 spend and 22% cannibalization loss, net result is -$1,874.50. Worse, post-promo demand dropped to 51 units/wk for 3 weeks, confirming heavy forward-buying — customers stockpiled, they didn't increase consumption."
}
"""


def analyze_promotion(promo: dict) -> dict:
    """Analyze a single promotion and return structured JSON."""
    user_prompt = (
        f"Analyze this promotion:\n\n"
        f"SKU: {promo['sku']}\n"
        f"Product: {promo['product_name']}\n"
        f"Category: {promo['category']}\n\n"
        f"BASELINE (no promo):\n"
        f"  Units/week: {promo['baseline_units_per_week']}\n"
        f"  Revenue/week: ${promo['baseline_revenue_week']:,.2f}\n"
        f"  Margin: {promo['baseline_margin']:.0%}\n\n"
        f"PROMOTION:\n"
        f"  Type: {promo['promo_type']}\n"
        f"  Duration: {promo['promo_duration_weeks']} weeks\n"
        f"  Lift: +{promo['promo_lift_pct']}%\n"
        f"  Units/week during promo: {promo['promo_units_per_week']}\n"
        f"  Margin during promo: {promo['promo_margin']:.0%}\n"
        f"  Total promo spend: ${promo['total_promo_cost']:,.2f}\n"
        f"  Cannibalization: {promo['cannibalization_pct']}% of lift from other category SKUs\n\n"
        f"POST-PROMO:\n"
        f"  Units in week after promo ended: {promo['post_promo_units_week1']}"
        f" (baseline was {promo['baseline_units_per_week']})\n\n"
        f"What should we do next?"
    )

    response = client.chat.completions.create(
        model="gpt-4o",
        min_completion_tokens=MIN_COMPLETION_TOKENS,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
    )

    result = json.loads(response.choices[0].message.content)
    result["_token_usage"] = {
        "prompt": response.usage.prompt_tokens,
        "completion": response.usage.completion_tokens,
        "total": response.usage.total_tokens,
    }
    return result


async def analyze_promotion_async(promo: dict) -> dict:
    """Async version — same logic, uses AsyncOpenAI for parallel execution."""
    user_prompt = (
        f"Analyze this promotion:\n\n"
        f"SKU: {promo['sku']}\n"
        f"Product: {promo['product_name']}\n"
        f"Category: {promo['category']}\n\n"
        f"BASELINE (no promo):\n"
        f"  Units/week: {promo['baseline_units_per_week']}\n"
        f"  Revenue/week: ${promo['baseline_revenue_week']:,.2f}\n"
        f"  Margin: {promo['baseline_margin']:.0%}\n\n"
        f"PROMOTION:\n"
        f"  Type: {promo['promo_type']}\n"
        f"  Duration: {promo['promo_duration_weeks']} weeks\n"
        f"  Lift: +{promo['promo_lift_pct']}%\n"
        f"  Units/week during promo: {promo['promo_units_per_week']}\n"
        f"  Margin during promo: {promo['promo_margin']:.0%}\n"
        f"  Total promo spend: ${promo['total_promo_cost']:,.2f}\n"
        f"  Cannibalization: {promo['cannibalization_pct']}% of lift from other category SKUs\n\n"
        f"POST-PROMO:\n"
        f"  Units in week after promo ended: {promo['post_promo_units_week1']}"
        f" (baseline was {promo['baseline_units_per_week']})\n\n"
        f"What should we do next?"
    )

    response = await async_client.chat.completions.create(
        model="gpt-4o",
        min_completion_tokens=MIN_COMPLETION_TOKENS,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
    )

    result = json.loads(response.choices[0].message.content)
    result["_token_usage"] = {
        "prompt": response.usage.prompt_tokens,
        "completion": response.usage.completion_tokens,
        "total": response.usage.total_tokens,
    }
    return result


print("analyze_promotion() and analyze_promotion_async() ready.")

In [ ]:
# ---------------------------------------------------------------------------
# Single call — one promotion analyzed
# ---------------------------------------------------------------------------
granola_promo = {
    "sku": "GRO-20847",
    "product_name": "Organic Honey Granola 16 oz",
    "category": "Breakfast / Cereal",
    "baseline_units_per_week": 142,
    "baseline_revenue_week": 1_136.00,
    "baseline_margin": 0.34,
    "promo_type": "BOGO 50% Off",
    "promo_duration_weeks": 3,
    "promo_lift_pct": 68.5,
    "promo_units_per_week": 239,
    "promo_margin": 0.18,
    "total_promo_cost": 1_850.00,
    "cannibalization_pct": 12.0,
    "post_promo_units_week1": 98,
}

result = analyze_promotion(granola_promo)
print(json.dumps(result, indent=2))

In [ ]:
# ---------------------------------------------------------------------------
# Parallel calls — 5 promotions analyzed concurrently with asyncio
#
# Instead of running 5 sequential calls (~30s), we fire them all at once
# and wait for all to finish (~6s). Same results, ~5x faster.
# ---------------------------------------------------------------------------
promotions = [
    {
        "sku": "GRO-20847",
        "product_name": "Organic Honey Granola 16 oz",
        "category": "Breakfast / Cereal",
        "baseline_units_per_week": 142,
        "baseline_revenue_week": 1_136.00,
        "baseline_margin": 0.34,
        "promo_type": "BOGO 50% Off",
        "promo_duration_weeks": 3,
        "promo_lift_pct": 68.5,
        "promo_units_per_week": 239,
        "promo_margin": 0.18,
        "total_promo_cost": 1_850.00,
        "cannibalization_pct": 12.0,
        "post_promo_units_week1": 98,
    },
    {
        "sku": "MEA-55120",
        "product_name": "Grass-Fed Ground Beef 1 lb",
        "category": "Meat / Poultry",
        "baseline_units_per_week": 310,
        "baseline_revenue_week": 2_790.00,
        "baseline_margin": 0.22,
        "promo_type": "$2 Off",
        "promo_duration_weeks": 1,
        "promo_lift_pct": 95.0,
        "promo_units_per_week": 605,
        "promo_margin": 0.08,
        "total_promo_cost": 500.00,
        "cannibalization_pct": 18.0,
        "post_promo_units_week1": 280,
    },
    {
        "sku": "SNK-33410",
        "product_name": "Sea Salt Kettle Chips 8 oz",
        "category": "Snacks",
        "baseline_units_per_week": 225,
        "baseline_revenue_week": 900.00,
        "baseline_margin": 0.41,
        "promo_type": "10% Off",
        "promo_duration_weeks": 2,
        "promo_lift_pct": 22.0,
        "promo_units_per_week": 275,
        "promo_margin": 0.33,
        "total_promo_cost": 200.00,
        "cannibalization_pct": 3.0,
        "post_promo_units_week1": 220,
    },
    {
        "sku": "BEV-41880",
        "product_name": "Cold Brew Coffee 12 oz 4-Pack",
        "category": "Beverages",
        "baseline_units_per_week": 64,
        "baseline_revenue_week": 768.00,
        "baseline_margin": 0.38,
        "promo_type": "Buy One Get One Free",
        "promo_duration_weeks": 2,
        "promo_lift_pct": 155.0,
        "promo_units_per_week": 163,
        "promo_margin": 0.05,
        "total_promo_cost": 1_200.00,
        "cannibalization_pct": 25.0,
        "post_promo_units_week1": 38,
    },
    {
        "sku": "PRO-62750",
        "product_name": "Organic Baby Spinach 5 oz",
        "category": "Produce",
        "baseline_units_per_week": 480,
        "baseline_revenue_week": 1_920.00,
        "baseline_margin": 0.45,
        "promo_type": "2 for $7 (reg $4.00 each)",
        "promo_duration_weeks": 1,
        "promo_lift_pct": 40.0,
        "promo_units_per_week": 672,
        "promo_margin": 0.32,
        "total_promo_cost": 150.00,
        "cannibalization_pct": 6.0,
        "post_promo_units_week1": 470,
    },
]


async def run_all():
    start = time.time()
    results = await asyncio.gather(
        *[analyze_promotion_async(p) for p in promotions]
    )
    elapsed = time.time() - start

    total_tokens = 0
    for r in results:
        sku = r.get("sku", "?")
        rec = r.get("recommendation", "?")
        profit = r.get("incremental_profit", 0)
        tokens = r["_token_usage"]["total"]
        total_tokens += tokens
        print(f"{sku:<12} → {rec:<13} | profit: ${profit:>+10,.2f} | tokens: {tokens:,}")

    print(f"\n--- Summary ---")
    print(f"Promotions analyzed: {len(results)}")
    print(f"Total tokens used:   {total_tokens:,}")
    print(f"Wall-clock time:     {elapsed:.1f}s (parallel)")

    return results


results = await run_all()